In [ ]:
# Install Tiny-SIMA into the active Jupyter kernel.
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault(
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    os.environ.get("TINY_SIMA_JAX_MEMORY_FRACTION", "0.60"),
)

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("Tiny-SIMA requires Python 3.11 or 3.12.")

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the Tiny-SIMA repository.")

cuda_override = os.environ.get("TINY_SIMA_USE_CUDA")
if cuda_override is None:
    has_cuda = (
        shutil.which("nvidia-smi") is not None
        and subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    )
else:
    has_cuda = cuda_override.lower() in {"1", "true", "yes"}

extras = "notebook,cuda12" if has_cuda else "notebook"
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--no-cache-dir",
    "--disable-pip-version-check",
    "-e",
    f"{repo_root}[{extras}]",
])

source_root = str(repo_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

print("Repository:", repo_root)
print("Source path added to kernel:", source_root)
print("CUDA detected:", has_cuda)


# Phase 4: local navigation and hierarchical skill chains

This notebook measures two generalization gaps before adding a VLM:

1. Can the executor solve randomized local-navigation scenarios with irrelevant objects?
2. Can skills be composed in one persistent Craftax world without resetting between requests?

The chain planner ranks each candidate by `utility × feasibility`, then replans after every success or failure. The scripted executor validates the environment and chain itself; the learned executor reveals the current policy ceiling.

By default, `QUICK_EVAL = False` runs the matched 2×2 chain ablation over 64 episodes using the same seed and horizon. Set it to `True` for a 16-episode iteration run. `RUN_LOCAL_SKILL_EVALUATIONS = False` skips the expensive local benchmark when it has already been measured. Learned fixed-plan chains are batched into one compiled JAX state machine; the scripted oracle and future arbitrary planners retain the generic Python evaluator.

In [ ]:
import jax
import matplotlib.pyplot as plt

from tiny_sima.envs import (
    SkillScenario,
    make_skill_environment,
    render_state,
    skills_for_scenario,
)
from tiny_sima.evaluation import (
    WOOD_TO_STONE_CHAIN,
    ScriptedSkillExecutor,
    evaluate_goal_conditioned_checkpoint,
    evaluate_learned_skill_chain_batched,
    evaluate_skill_chain,
    evaluate_skill_policy,
    learned_skill_executor_from_checkpoint,
)
from tiny_sima.goals import SkillId

print("JAX devices:", jax.devices())


In [ ]:
CHECKPOINT_RUN = "skill-ppo-local-1m"
CHECKPOINT = repo_root / "runs" / CHECKPOINT_RUN / "checkpoints" / "final"

# Learned fixed-plan chains are JAX-batched. Scripted and arbitrary-planner
# paths remain sequential. Quick mode is intended for rapid iteration.
QUICK_EVAL = False
RUN_LOCAL_SKILL_EVALUATIONS = False

NUM_LOCAL_EPISODES = 32 if QUICK_EVAL else 128
NUM_SCRIPTED_LOCAL_EPISODES = 8 if QUICK_EVAL else 32
NUM_CHAIN_EPISODES = 4 if QUICK_EVAL else 8
NUM_DIAGNOSTIC_CHAIN_EPISODES = 16 if QUICK_EVAL else 64
LOCAL_HORIZON = 32
CHAIN_SKILL_HORIZON = 32 if QUICK_EVAL else 64

print("Checkpoint:", CHECKPOINT)
print("Checkpoint exists:", CHECKPOINT.exists())


In [ ]:
local_environment = make_skill_environment(
    SkillId.COLLECT_WOOD,
    horizon=LOCAL_HORIZON,
    scenario=SkillScenario.LOCAL_NAVIGATION,
    min_distance=2,
    max_distance=4,
    distractor_count=6,
)
local_reset = local_environment.reset(jax.random.PRNGKey(7))
pixels = render_state(
    local_reset.state.env_state,
    local_environment.bundle.spec.name,
    block_pixel_size=16,
)

plt.figure(figsize=(7, 5))
plt.imshow(pixels)
plt.axis("off")
plt.title("Randomized local-navigation scenario")
plt.show()


In [ ]:
# Run only the scripted policy. evaluate_skill_suite() also evaluates a
# random baseline that is unnecessary after validating the scenarios. Like the
# learned cell below, it covers the six skills the checkpoint was trained on.
if RUN_LOCAL_SKILL_EVALUATIONS:
    scripted_local = [
        evaluate_skill_policy(
            skill_id,
            policy="scripted",
            num_episodes=NUM_SCRIPTED_LOCAL_EPISODES,
            horizon=LOCAL_HORIZON,
            seed=11 + index,
            scenario=SkillScenario.LOCAL_NAVIGATION,
            min_distance=2,
            max_distance=4,
            distractor_count=6,
        )
        for index, skill_id in enumerate(
            skills_for_scenario(SkillScenario.LOCAL_NAVIGATION)
        )
    ]
    for result in scripted_local:
        print(
            f'{result["skill_id"]:22s} '
            f'scripted={result["success_rate"]:.3f} '
            f'length={result["mean_episode_length"]:.2f}'
        )
else:
    print("Skipping repeated scripted local-skill validation.")


In [ ]:
if RUN_LOCAL_SKILL_EVALUATIONS and CHECKPOINT.exists():
    learned_local = evaluate_goal_conditioned_checkpoint(
        CHECKPOINT,
        num_episodes_per_skill=NUM_LOCAL_EPISODES,
        seed=21,
        deterministic=True,
        scenario=SkillScenario.LOCAL_NAVIGATION,
        horizon=LOCAL_HORIZON,
        min_distance=2,
        max_distance=4,
        distractor_count=6,
    )
    for result in learned_local:
        print(
            f'{result["skill_id"]:22s} '
            f'success={result["success_rate"]:.3f} '
            f'timeout={result["feasible_timeout_rate"]:.3f} '
            f'length={result["mean_episode_length"]:.2f}'
        )
elif not RUN_LOCAL_SKILL_EVALUATIONS:
    print("Skipping repeated learned local-skill evaluation.")
else:
    print("Checkpoint not found; skipping learned local-navigation evaluation.")


In [ ]:
# This validates persistent state, chain preconditions, outcome accounting,
# and closed-loop replanning independently of the learned policy.
oracle_chain = evaluate_skill_chain(
    ScriptedSkillExecutor(),
    num_episodes=NUM_CHAIN_EPISODES,
    seed=31,
    skill_horizon=CHAIN_SKILL_HORIZON,
)
{key: value for key, value in oracle_chain.items() if key != "episodes"}


In [ ]:
if CHECKPOINT.exists():
    learned_executor = learned_skill_executor_from_checkpoint(
        CHECKPOINT,
        deterministic=True,
    )
    learned_chain = evaluate_learned_skill_chain_batched(
        learned_executor,
        num_episodes=NUM_DIAGNOSTIC_CHAIN_EPISODES,
        seed=41,
        skill_horizon=CHAIN_SKILL_HORIZON,
    )
    print(learned_chain)
else:
    print("Checkpoint not found; skipping learned chain evaluation.")


## Diagnostic 1: stochastic execution

This reruns the default chain with sampled actions. Improvement over deterministic execution indicates that the policy assigns probability to useful recovery or exploration actions even when its highest-probability action loops or stalls.

In [ ]:
if CHECKPOINT.exists():
    stochastic_executor = learned_skill_executor_from_checkpoint(
        CHECKPOINT,
        deterministic=False,
    )
    stochastic_chain = evaluate_learned_skill_chain_batched(
        stochastic_executor,
        num_episodes=NUM_DIAGNOSTIC_CHAIN_EPISODES,
        seed=41,
        skill_horizon=CHAIN_SKILL_HORIZON,
    )
    print(stochastic_chain)
else:
    print("Checkpoint not found; skipping stochastic chain diagnostic.")


## Diagnostic 2: collect all wood before placing the table

This uses the same objective and persistent environment but changes the reference sequence to collect all three wood before modifying the map with a crafting table. It tests whether the original post-table `collect_wood` boundary is the principal distribution shift.

In [ ]:
collect_first_skills = (
    SkillId.COLLECT_WOOD,
    SkillId.COLLECT_WOOD,
    SkillId.COLLECT_WOOD,
    SkillId.PLACE_TABLE,
    SkillId.CRAFT_WOOD_PICKAXE,
    SkillId.COLLECT_STONE,
)
collect_first_chain = WOOD_TO_STONE_CHAIN._replace(
    name="wood_to_stone_collect_first",
    reference_skills=collect_first_skills,
)
if CHECKPOINT.exists():
    collect_first_result = evaluate_learned_skill_chain_batched(
        learned_executor,
        chain=collect_first_chain,
        num_episodes=NUM_DIAGNOSTIC_CHAIN_EPISODES,
        seed=41,
        skill_horizon=CHAIN_SKILL_HORIZON,
    )
    print(collect_first_result)
else:
    print("Checkpoint not found; skipping alternative-plan diagnostic.")


## Diagnostic 3: stochastic collect-first execution

This fills the final cell of the matched 2×2 comparison by combining sampled actions with the collect-first plan.

In [ ]:
if CHECKPOINT.exists():
    collect_first_stochastic_result = evaluate_learned_skill_chain_batched(
        stochastic_executor,
        chain=collect_first_chain,
        num_episodes=NUM_DIAGNOSTIC_CHAIN_EPISODES,
        seed=41,
        skill_horizon=CHAIN_SKILL_HORIZON,
    )
    print(collect_first_stochastic_result)
else:
    print("Checkpoint not found; skipping stochastic collect-first diagnostic.")


## Matched 2×2 comparison

All four learned evaluations use identical episode counts, environment seeds, skill horizons, retry limits, and distractor settings. Only plan order and action selection differ.

In [ ]:
if CHECKPOINT.exists():
    matched_results = (
        ("default", "deterministic", learned_chain),
        ("default", "stochastic", stochastic_chain),
        ("collect_first", "deterministic", collect_first_result),
        ("collect_first", "stochastic", collect_first_stochastic_result),
    )
    print(
        f'{"plan":<16} {"policy":<14} {"success":>8} '
        f'{"skills":>8} {"steps":>8} {"seconds":>9} failures'
    )
    for plan_name, policy_name, result in matched_results:
        print(
            f'{plan_name:<16} {policy_name:<14} '
            f'{result["success_rate"]:8.3f} '
            f'{result["mean_completed_skills"]:8.2f} '
            f'{result["mean_environment_steps"]:8.1f} '
            f'{result["total_seconds_including_compile"]:9.2f} '
            f'{result["final_failed_skill_counts"]}'
        )
else:
    print("Checkpoint not found; skipping matched ablation summary.")


## How to interpret the comparison

- Scripted local failure means the scenario generator is malformed.
- Learned local failure isolates navigation or distractor generalization.
- Scripted chain failure means the persistent chain or preconditions are wrong.
- Learned chain failure with a successful scripted chain measures executor compounding.
- Stochastic improvement indicates useful actions exist below the deterministic argmax.
- Alternative-plan improvement implicates the post-table skill boundary; no improvement points toward target visibility, distance, or broader persistent-state shift.

The future VLM replaces only the utility scorer. Symbolic or learned feasibility remains a separate factor, and every skill outcome is returned to the planner before it makes the next decision.